In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.data.player_ids import load_player_ids, display_name

df = load_all_snapshots(seasons=[2024])
f = add_discipline_flags(df)
f = f[f["pitch_type"].notna()]

# Count buckets that matter tactically
def count_bucket(balls, strikes):
    if strikes == 2:
        return "two_strike"
    if balls > strikes:
        return "behind"      # pitcher behind
    if strikes > balls:
        return "ahead"
    return "even"

f["count_bucket"] = [count_bucket(b, s) for b, s in
                     zip(f["balls"].fillna(0).astype(int),
                         f["strikes"].fillna(0).astype(int))]

usage = (
    f.groupby(["pitcher", "count_bucket", "pitch_type"]).size()
    .rename("n").reset_index()
)
totals = f.groupby(["pitcher", "count_bucket"]).size().rename("total")
usage = usage.join(totals, on=["pitcher", "count_bucket"])
usage["pct"] = usage["n"] / usage["total"]

lg_usage = (
    f.groupby(["count_bucket", "pitch_type"]).size()
    / f.groupby("count_bucket").size()
).rename("lg_pct")

print(lg_usage.unstack().round(3).to_string())

pitch_type       CH   CS     CU     EP     FA     FC     FF   FO     FS     KC     KN   PO   SC     SI     SL     ST     SV   UN
count_bucket                                                                                                                    
ahead         0.130  0.0  0.060  0.001  0.001  0.083  0.285  0.0  0.040  0.017  0.002  0.0  0.0  0.142  0.149  0.083  0.006  NaN
behind        0.099  0.0  0.039  0.001  0.001  0.101  0.344  0.0  0.021  0.010  0.001  0.0  0.0  0.201  0.129  0.049  0.004  0.0
even          0.086  0.0  0.069  0.001  0.001  0.090  0.315  0.0  0.023  0.018  0.002  0.0  0.0  0.178  0.142  0.070  0.006  NaN
two_strike    0.112  0.0  0.069  0.000  0.000  0.059  0.320  0.0  0.042  0.022  0.002  0.0  0.0  0.111  0.165  0.090  0.005  0.0


In [2]:
sw = f[f["is_swing"]]
oz = f[~f["in_zone"]]

outcomes = pd.DataFrame({
    "pitches": f.groupby(["pitcher", "pitch_type"]).size(),
    "swings": sw.groupby(["pitcher", "pitch_type"]).size(),
    "whiffs": sw.groupby(["pitcher", "pitch_type"])["is_whiff"].sum(),
    "zone_pct": f.groupby(["pitcher", "pitch_type"])["in_zone"].mean(),
    "oz_pitches": oz.groupby(["pitcher", "pitch_type"]).size(),
    "oz_swings": oz.groupby(["pitcher", "pitch_type"])["is_swing"].sum(),
    "velo": f.groupby(["pitcher", "pitch_type"])["release_speed"]
             .apply(lambda s: pd.to_numeric(s, errors="coerce").mean()),
}).fillna(0)
outcomes["whiff_pct"] = outcomes["whiffs"] / outcomes["swings"].replace(0, np.nan)
outcomes["chase_pct"] = outcomes["oz_swings"] / outcomes["oz_pitches"].replace(0, np.nan)

lg_out = pd.DataFrame({
    "lg_whiff": sw.groupby("pitch_type")["is_whiff"].mean(),
    "lg_chase": oz.groupby("pitch_type")["is_swing"].mean(),
    "lg_zone": f.groupby("pitch_type")["in_zone"].mean(),
    "lg_velo": f.groupby("pitch_type")["release_speed"]
                .apply(lambda s: pd.to_numeric(s, errors="coerce").mean()),
})
print(lg_out.round(3).to_string())

            lg_whiff  lg_chase  lg_zone  lg_velo
pitch_type                                      
CH             0.293     0.337    0.390   85.451
CS             0.125     0.308    0.361   58.598
CU             0.296     0.290    0.436   79.370
EP             0.020     0.285    0.360   50.430
FA             0.098     0.216    0.471   68.063
FC             0.213     0.272    0.516   89.485
FF             0.189     0.239    0.554   94.291
FO             0.272     0.427    0.360   82.079
FS             0.327     0.359    0.372   86.526
KC             0.329     0.322    0.410   81.774
KN             0.274     0.250    0.427   77.639
PO               NaN     0.000    0.019   91.306
SC             0.232     0.226    0.392   80.873
SI             0.117     0.246    0.568   93.290
SL             0.323     0.321    0.453   85.818
ST             0.297     0.308    0.442   81.993
SV             0.271     0.292    0.431   81.341
UN               NaN     0.000    0.000   64.650


In [3]:
MIN_PITCHES = 100
MIN_SWINGS = 40

def pitcher_report(pitcher_id, names=None):
    name = names.get(pitcher_id, str(pitcher_id)) if names is not None else str(pitcher_id)
    lines = [f"# {name}"]

    if pitcher_id not in outcomes.index.get_level_values(0):
        return "\n".join(lines + ["", "INSUFFICIENT SAMPLE"])

    o = outcomes.loc[pitcher_id]
    o = o[o["pitches"] >= MIN_PITCHES].join(lg_out)
    if len(o) == 0:
        return "\n".join(lines + ["", "INSUFFICIENT SAMPLE"])

    total = o["pitches"].sum()
    o = o.assign(usage=o["pitches"] / total)

    lines.append("\n## Arsenal")
    for pt, r in o.sort_values("usage", ascending=False).iterrows():
        velo_gap = r["velo"] - r["lg_velo"]
        lines.append(
            f"  {pt}: {r['usage']:.1%} usage, {r['velo']:.1f} mph "
            f"({velo_gap:+.1f} vs league), zone {r['zone_pct']:.1%}")

    lines.append("\n## Effectiveness")
    for pt, r in o.sort_values("usage", ascending=False).iterrows():
        if r["swings"] < MIN_SWINGS:
            lines.append(f"  {pt}: INSUFFICIENT SAMPLE ({int(r['swings'])} swings)")
            continue
        wgap = r["whiff_pct"] - r["lg_whiff"]
        tag = ""
        if wgap > 0.05:
            tag = "  <-- plus pitch"
        elif wgap < -0.05:
            tag = "  <-- below league"
        lines.append(
            f"  {pt}: whiff {r['whiff_pct']:.1%} vs {r['lg_whiff']:.1%} "
            f"({wgap:+.1%}), {int(r['swings'])} swings{tag}")

    # Two-strike selection
    ts = usage[(usage["pitcher"] == pitcher_id)
               & (usage["count_bucket"] == "two_strike")]
    ts = ts[ts["n"] >= 20].sort_values("pct", ascending=False)
    lines.append("\n## Two-strike selection")
    if len(ts) == 0:
        lines.append("  INSUFFICIENT SAMPLE")
    else:
        for _, r in ts.head(4).iterrows():
            lg = lg_usage.get(("two_strike", r["pitch_type"]), np.nan)
            lines.append(f"  {r['pitch_type']}: {r['pct']:.1%} "
                         f"(league {lg:.1%}, {int(r['n'])} pitches)")

    return "\n".join(lines)

# Skubal, Sale, Wheeler
ids = load_player_ids([669373, 519242, 554430])
names = display_name(ids)
print(pitcher_report(669373, names))

# Skubal, Tarik

## Arsenal
  FF: 33.2% usage, 96.9 mph (+2.6 vs league), zone 60.1%
  CH: 27.1% usage, 86.3 mph (+0.9 vs league), zone 46.4%
  SI: 20.6% usage, 96.6 mph (+3.3 vs league), zone 61.1%
  SL: 14.9% usage, 88.7 mph (+2.9 vs league), zone 53.7%
  CU: 4.2% usage, 78.5 mph (-0.9 vs league), zone 55.0%

## Effectiveness
  FF: whiff 23.2% vs 18.9% (+4.3%), 499 swings
  CH: whiff 43.9% vs 29.3% (+14.6%), 456 swings  <-- plus pitch
  SI: whiff 11.3% vs 11.7% (-0.4%), 275 swings
  SL: whiff 32.1% vs 32.3% (-0.2%), 190 swings
  CU: whiff 17.5% vs 29.6% (-12.1%), 40 swings  <-- below league

## Two-strike selection
  FF: 34.9% (league 32.0%, 306 pitches)
  CH: 30.3% (league 11.2%, 266 pitches)
  SI: 20.8% (league 11.1%, 182 pitches)
  SL: 12.4% (league 16.5%, 109 pitches)


In [4]:
def hitting_approach(pitcher_id, min_pitches=100, min_swings=40):
    """What a hitter should do against this pitcher.

    Mirror of the batter report, but a pitcher CHOOSES what to throw, so
    usage and count tendency matter as much as pitch quality.
    """
    if pitcher_id not in outcomes.index.get_level_values(0):
        return ["INSUFFICIENT SAMPLE"]

    o = outcomes.loc[pitcher_id]
    o = o[o["pitches"] >= min_pitches].join(lg_out)
    if len(o) == 0:
        return ["INSUFFICIENT SAMPLE"]

    o = o.assign(usage=o["pitches"] / o["pitches"].sum(),
                 wgap=o["whiff_pct"] - o["lg_whiff"])

    recs = []

    # The out pitch: most used with two strikes, relative to league
    ts = usage[(usage["pitcher"] == pitcher_id)
               & (usage["count_bucket"] == "two_strike") & (usage["n"] >= 20)]
    if len(ts) > 0:
        ts = ts.copy()
        ts["lg"] = [lg_usage.get(("two_strike", pt), np.nan) for pt in ts["pitch_type"]]
        ts["over"] = ts["pct"] - ts["lg"]
        top = ts.nlargest(1, "over").iloc[0]
        if top["over"] > 0.05:
            pt = top["pitch_type"]
            wg = o.loc[pt, "wgap"] if pt in o.index else np.nan
            detail = f", whiffs {wg:+.1%} vs league" if pd.notna(wg) else ""
            recs.append(
                f"OUT PITCH {pt}: {top['pct']:.1%} with two strikes vs league "
                f"{top['lg']:.1%}{detail} ({int(top['n'])} pitches)")

    # Weak offerings
    weak = o[(o["wgap"] < -0.05) & (o["swings"] >= min_swings)]
    for pt, r in weak.sort_values("wgap").iterrows():
        recs.append(
            f"Hunt {pt}: whiffs {r['wgap']:+.1%} below league, "
            f"{r['usage']:.1%} usage ({int(r['swings'])} swings)")

    # Pitches with low swings — explicitly unmeasured
    unmeasured = o[o["swings"] < min_swings]
    for pt, r in unmeasured.iterrows():
        recs.append(f"{pt}: NOT MEASURED ({int(r['swings'])} swings, "
                    f"need {min_swings}) — {r['usage']:.1%} usage")

    # Zone tendency by count
    behind = usage[(usage["pitcher"] == pitcher_id)
                   & (usage["count_bucket"] == "behind")]
    if len(behind) > 0:
        fb = behind[behind["pitch_type"].isin(["FF", "SI", "FC"])]["pct"].sum()
        if fb > 0.55:
            recs.append(f"Behind in the count: {fb:.1%} fastballs — sit on it")

    return recs or ["No exploitable tendencies at these sample sizes"]

for pid in [669373, 519242, 554430]:
    print(f"=== {names.get(pid, pid)} ===")
    for r in hitting_approach(pid):
        print(f"  - {r}")
    print()

=== Skubal, Tarik ===
  - OUT PITCH CH: 30.3% with two strikes vs league 11.2%, whiffs +14.6% vs league (266 pitches)
  - Hunt CU: whiffs -12.1% below league, 4.2% usage (40 swings)
  - Behind in the count: 61.8% fastballs — sit on it

=== Sale, Chris ===
  - OUT PITCH SL: 44.9% with two strikes vs league 16.5%, whiffs +8.8% vs league (427 pitches)
  - Hunt CH: whiffs -6.1% below league, 14.2% usage (207 swings)

=== Wheeler, Zack ===
  - OUT PITCH CU: 14.9% with two strikes vs league 6.9%, whiffs +6.4% vs league (150 pitches)
  - Hunt FC: whiffs -5.8% below league, 9.6% usage (154 swings)
  - Behind in the count: 79.4% fastballs — sit on it

